# Week 5 -- Function 5: Bayesian Optimisation (4D)

In [ ]:
# Cell 2: Imports
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

print('All libraries loaded successfully.')

In [ ]:
# Cell 3: Load data and inspect
# Function 5: Chemical process yield (unimodal), 4D input, Maximise

X = np.load('../data/function_5/initial_inputs.npy')
Y = np.load('../data/function_5/initial_outputs.npy')

print(f'Input  shape : {X.shape}   (n_samples x n_dimensions)')
print(f'Output shape : {Y.shape}  (n_samples,)')
print()

# Sort descending by Y value
X_list = list(X)
Y_list = list(Y)
pairs = sorted(zip(Y_list, X_list), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 78)
print('  All observations (sorted descending by Y)')
print('=' * 78)
for i, (y_val, x_val) in enumerate(zip(Y_sorted, X_sorted)):
    marker = '  <-- best' if i == 0 else ''
    print(f'  [{i+1:2d}]  X = [{x_val[0]:.6f}, {x_val[1]:.6f}, {x_val[2]:.6f}, {x_val[3]:.6f}]   Y = {y_val:+.4f}{marker}')
print('=' * 78)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
print(f'\n  Best Y*  = {best_Y:.4f}')
print(f'  Best X*  = [{best_X[0]:.8f}, {best_X[1]:.8f}, {best_X[2]:.8f}, {best_X[3]:.8f}]')

In [ ]:
# Cell 4: Fit GP with log-transformed Y

# Log-transform to handle extreme scale differences across observations
Y_fit = np.log(np.abs(Y) + 1e-300)

# Fixed RBF kernel (course style -- no ConstantKernel, no optimisation)
kernel = RBF(length_scale=0.1, length_scale_bounds='fixed')
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10)
gp.fit(X, Y_fit)

print('=' * 62)
print('  GP Fitting Results')
print('=' * 62)
print(f'  Kernel                   : {gp.kernel_}')
print(f'  Log-marginal-likelihood  : {gp.log_marginal_likelihood_value_:.4f}')
print()

# Sanity check: predict at best known point
pred_mean, pred_std = gp.predict(best_X.reshape(1, -1), return_std=True)
actual_log = np.log(np.abs(best_Y) + 1e-300)
print('  Sanity check at best known X*:')
print(f'    X*                     = [{best_X[0]:.6f}, {best_X[1]:.6f}, {best_X[2]:.6f}, {best_X[3]:.6f}]')
print(f'    GP predicted mean      = {pred_mean[0]:.4f}  (log-space)')
print(f'    Actual log(|Y*|)       = {actual_log:.4f}  (log-space)')
print(f'    GP predicted std       = {pred_std[0]:.8f}')
print('=' * 62)

In [ ]:
# Cell 5: UCB acquisition over random search (20,000 points)
# 4D grid search (10^4 = 10,000) is feasible but random search gives denser coverage

np.random.seed(42)
X_grid = np.random.uniform(0, 1, size=(20000, 4))  # shape (20000, 4)

# GP predictions across the random grid
post_mean, post_std = gp.predict(X_grid, return_std=True)

# UCB acquisition: mean + beta * std
beta = 2.5  #Higher beta = more exploration at early stage (Week 1, sparse data)
acquisition = post_mean + beta * post_std  # shape (20000,)

# Next query = grid point with highest UCB
best_idx = np.argmax(acquisition)
next_x   = X_grid[best_idx]
next_acq = acquisition[best_idx]

# Portal submission string
portal_string = f'{next_x[0]:.6f}-{next_x[1]:.6f}-{next_x[2]:.6f}-{next_x[3]:.6f}'

print('=' * 62)
print('  UCB Acquisition  (beta = 2.5, random search 20,000 pts)')
print('=' * 62)
print(f'  Grid size            : {len(X_grid)} points (random uniform)')
print(f'  Max UCB value        : {next_acq:.4f}')
print(f'  Next query point     : [{next_x[0]:.6f}, {next_x[1]:.6f}, {next_x[2]:.6f}, {next_x[3]:.6f}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 62)

In [ ]:
# Cell 7: Summary

print('=' * 62)
print('  SUMMARY -- Bayesian Optimisation Results')
print('=' * 62)
print(f'  Function             : 4D Chemical Process Yield (unimodal)')
print(f'  Objective            : Maximise')
print(f'  Kernel               : RBF(length_scale=0.1, fixed)')
print(f'  Acquisition function : UCB  (beta = 2.5)')
print(f'  Y transform          : log(|Y| + 1e-300)')
print(f'  Grid search          : Random uniform (20,000 points)')
print()
print(f'  Current best X*      : [{best_X[0]:.6f}, {best_X[1]:.6f}, {best_X[2]:.6f}, {best_X[3]:.6f}]')
print(f'  Current best Y*      : {best_Y:.4f}')
print()
print(f'  Next query point     : [{next_x[0]:.6f}, {next_x[1]:.6f}, {next_x[2]:.6f}, {next_x[3]:.6f}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 62)

## Week 1 -- Reflection

**Function**: F5 -- Chemical Process Yield Optimisation (4D)  
**Domain context**: Typically unimodal -- single peak where yield is maximised

### Query
- **Method**: GP + UCB (beta=2.5), RBF kernel (length_scale=0.1, fixed), 10×10×10×10 meshgrid
- **Query type**: Exploration -- first submission, high uncertainty everywhere
- **Input submitted**: [0.209005, 0.838746, 0.859156, 0.882439]
- **Output received**: 984.409363

### What this result taught us
Strong yield on first query (y=984.41). The function is likely unimodal; the peak may be close to x1~0.21, x2-x4~0.86-0.88.

### Strategy for Week 2
Slight reduction in beta to 2.5 to refine around the x1~0.21, x2-x4~0.87 region; the function appears unimodal.